# GitAgent — Overview & Example Usage

GitAgent is a git-backed memory management system for agent sessions. It's part of a personal learning project (`PersonalWorkspace`) whose goal is to master LLMs, RAG, and agentic systems by dogfooding self-built tools.

**Two separate layers:**
1. **Build-time** — Claude Code writes and iterates on the GitAgent codebase itself (`gitagent/cli.py`, `gitagent/tools.py`, `gitagent/visualize.py`).
2. **Run-time** — GitAgent calls a **local Ollama model** to do its work (currently just branch-close summarization). Claude is never involved at run time.

**Core idea:** use real `git` as the substrate for agent memory, instead of reinventing branching/versioning. A "branch" of work gets a real `git branch` that actually diverges — its `MEMORY.md` only ever lives in that branch's own history. Branches can nest: you can open one from another open branch instead of always from `main`. Only a *top-level* branch (one opened straight off `main`) gets a row in `STATE_TRACKER.md` — nesting can go arbitrarily deep underneath it without `STATE_TRACKER.md` growing a row per branch; the detail lives one hop away, in that top-level branch's own `MEMORY.md`. This notebook walks through all four tools end to end, including nesting, against a disposable sandbox repo, so it's safe to re-run.

## Directory layout

```text
workspace/
├── STATE_TRACKER.md          # main line — one row per TOP-LEVEL branch only, read/written on `main`
├── branch_graph.html         # generated by `gitagent graph` — not committed, regenerate anytime
├── branches/
│   ├── <branch-name>/
│   │   └── MEMORY.md         # scoped working notes - exists only on that branch's own history;
│   │                         # has its own `## Sub-Branches` section indexing every branch nested under it
│   └── archived/
│       └── <branch-name>/
│           └── MEMORY.md     # moved here on close, on the *base* branch — raw log, never deleted
└── gitagent/
    ├── cli.py                # argparse CLI: open-branch / update-branch / close-branch / graph
    ├── tools.py               # open_branch / update_branch / close_branch
    └── visualize.py           # render_branch_graph
```

## The tools

| Tool | Git operation | LLM call? | Ends checked out on | What it does |
|---|---|---|---|---|
| `open_branch(name, description, base="main")` | `git checkout -b name base` | No | `name` | Creates a real branch off `base`. If `base` is `main`, inserts an `Active` row into `STATE_TRACKER.md` on `main`. Otherwise `name` is nested: no `STATE_TRACKER.md` row at all - instead an `Active` line is appended to the *root* branch's (the ancestor that itself was opened off `main`) `## Sub-Branches` section. |
| `update_branch(name, note)` | commit on `name` | No | `name` | Checks out `name`, appends a timestamped bullet to its own `MEMORY.md` Decisions Log, commits it — the "commit equivalent" of a working note. |
| `close_branch(name)` | `git merge` + `git mv` + commit | **Yes** — local Ollama | `base` | Summarizes `MEMORY.md` via Ollama and merges `name` into its recorded `base` (a **real** merge - `name`'s commits actually diverged). If `name` was top-level, marks its `STATE_TRACKER.md` row `Completed` with that summary. If nested, instead flips its line under the root's `## Sub-Branches` to `Completed` with that summary. Either way archives (moves, never deletes) the memory file onto `base`. |
| `render_branch_graph(output_path=None)` | reads `git log --all` | No | *(read-only)* | Renders every commit/branch as a standalone, self-contained HTML page (lane-colored graph + scrollable log). |

**Why this scales:** without this, every branch ever opened - no matter how deep the nesting - would get its own permanent `STATE_TRACKER.md` row, which turns it into a burden as a project grows. Instead `STATE_TRACKER.md` stays at exactly one row per top-level side branch, and the full nested history is flattened into that branch's own `## Sub-Branches` section (every descendant at any depth, not just direct children) - one line per branch, not one line per note. By the time a top-level branch itself closes, its `Sub-Branches` section already holds every descendant's own summary, so the single `STATE_TRACKER.md` row Ollama generates for it is a genuine roll-up of the whole tree, not just that branch's own top-level notes.

`STATE_TRACKER.md` is only ever read/written while checked out on `main`. Its *working-tree copy* on any other branch is just whatever `main` looked like when that branch forked - to see the current, canonical table from anywhere, use `git show main:STATE_TRACKER.md` rather than trusting the file on disk unless you're actually on `main`.

## Setup: a disposable sandbox repo

The cells below run the real `gitagent.tools` / `gitagent.visualize` functions, but against a throwaway git repo in a temp directory (not this project's own repo), so you can re-run this notebook freely. We do this by importing the modules and pointing their module-level path constants at the sandbox before calling anything.

In [1]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from gitagent import tools, visualize

sandbox = Path(tempfile.mkdtemp(prefix="gitagent_demo_"))
print(f"Sandbox repo: {sandbox}")

subprocess.run(["git", "init", "-q", "-b", "main"], cwd=sandbox, check=True)
subprocess.run(["git", "config", "user.email", "demo@example.com"], cwd=sandbox, check=True)
subprocess.run(["git", "config", "user.name", "GitAgent Demo"], cwd=sandbox, check=True)

state_tracker = sandbox / "STATE_TRACKER.md"
state_tracker.write_text(
    "# Demo State Tracker\n\n"
    "## Side Branches (Features & Quests)\n\n"
    "| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |\n"
    "|---|---|---|---|---|\n",
    encoding="utf-8",
)
subprocess.run(["git", "add", "STATE_TRACKER.md"], cwd=sandbox, check=True)
subprocess.run(["git", "commit", "-q", "-m", "Initial commit"], cwd=sandbox, check=True)

# Point gitagent.tools and gitagent.visualize at the sandbox instead of this repo.
tools.WORKSPACE_ROOT = sandbox
tools.BRANCHES_DIR = sandbox / "branches"
tools.ARCHIVED_BRANCHES_DIR = tools.BRANCHES_DIR / "archived"
tools.STATE_TRACKER_PATH = state_tracker
visualize.WORKSPACE_ROOT = sandbox

print("Ready.")

Sandbox repo: C:\Users\user\AppData\Local\Temp\gitagent_demo_yxvtla6g
Ready.


## 1. `open_branch` — start scoped work (and nest it)

First a normal top-level branch off `main`. Then a *nested* branch opened off that branch instead of `main` - this is the piece that didn't exist before: `open_branch`'s `base` argument. Notice it does **not** add a second `STATE_TRACKER.md` row - it shows up as a line in the parent's own `MEMORY.md` instead.

In [3]:
parent_memory = tools.open_branch(
    "search-endpoint",
    "Add a /search endpoint backed by the new embeddings index.",
)
print(parent_memory)
print(parent_memory.read_text(encoding="utf-8"))

GitAgentError: branch already exists: search-endpoint

In [4]:
# Nested: opened FROM search-endpoint, not from main.
child_memory = tools.open_branch(
    "search-endpoint-ranking",
    "Work out the ranking/scoring formula for search results.",
    base="search-endpoint",
)
print(child_memory.read_text(encoding="utf-8"))

# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Active

## Decisions Log

## Sub-Branches

## Open Questions



In [5]:
# STATE_TRACKER.md is only trustworthy read straight from main - see the
# note above. Still exactly ONE row here, even after opening the nested
# ranking branch - it doesn't get one of its own.
canonical_tracker = subprocess.run(
    ["git", "show", "main:STATE_TRACKER.md"], cwd=sandbox, capture_output=True, text=True, check=True
).stdout
print(canonical_tracker)

# Instead, look at search-endpoint's own MEMORY.md - the ranking branch shows
# up there, under Sub-Branches.
print(parent_memory.read_text(encoding="utf-8"))

# Demo State Tracker

## Side Branches (Features & Quests)

| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |
|---|---|---|---|---|
| Branch-001 | search-endpoint | Add a /search endpoint backed by the new embeddings index. | Active | |

# Branch Memory: search-endpoint

## Branched From
main

## Goal
Add a /search endpoint backed by the new embeddings index.

## Status
Active

## Decisions Log

## Sub-Branches
- **search-endpoint-ranking** — Active — Work out the ranking/scoring formula for search results.

## Open Questions



## 2. `update_branch` — log notes as you go

Each call checks out the target branch, appends one timestamped bullet to the Decisions Log, and makes its own commit there — no LLM call involved, it's pure structured logging.

In [6]:
tools.update_branch(
    "search-endpoint",
    "Chose FAISS over a hosted vector DB — no external dependency for a local-first tool.",
)
tools.update_branch(
    "search-endpoint-ranking",
    "Cosine similarity on the raw embedding, no re-ranking pass for v1.",
)
tools.update_branch(
    "search-endpoint-ranking",
    "Added a basic integration test for the ranking function.",
)
print(child_memory.read_text(encoding="utf-8"))

# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Active

## Decisions Log
- [2026-08-30 18:26 UTC] Cosine similarity on the raw embedding, no re-ranking pass for v1.
- [2026-08-30 18:26 UTC] Added a basic integration test for the ranking function.

## Sub-Branches

## Open Questions



## 3. `close_branch` — summarize, merge, archive

This step calls a **local Ollama model** (default `llama3.1:8b` at `http://localhost:11434`, override with the `GITAGENT_OLLAMA_MODEL` / `GITAGENT_OLLAMA_HOST` env vars) to compress the branch's `MEMORY.md` into 2–3 sentences. The prompt explicitly forbids markdown, `|`, and any preamble like "Here is a summary" — small instruct models tend to add one anyway, so double-check the generated text before trusting it verbatim. These cells need `ollama serve` running with that model pulled; they fail gracefully if it isn't reachable.

Closing the **nested** branch first is the interesting part: it merges into `search-endpoint` (its recorded base), *not* `main`, and its Sub-Branches line on `search-endpoint` flips from Active to Completed - `STATE_TRACKER.md` is untouched.

In [7]:
try:
    archived_child = tools.close_branch("search-endpoint-ranking")
    current = subprocess.run(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=sandbox, capture_output=True, text=True, check=True
    ).stdout.strip()
    print(f"Archived to: {archived_child}")
    print(f"Ended checked out on: {current}  (merged into its base, not main)")
    print("-" * 60)
    print(archived_child.read_text(encoding="utf-8"))
    print("-" * 60)
    print("search-endpoint's Sub-Branches line for it is now Completed:")
    print(parent_memory.read_text(encoding="utf-8"))
except tools.GitAgentError as exc:
    print(f"close_branch failed (is Ollama running? `ollama serve`): {exc}")

Archived to: C:\Users\user\AppData\Local\Temp\gitagent_demo_yxvtla6g\branches\archived\search-endpoint-ranking\MEMORY.md
Ended checked out on: search-endpoint  (merged into its base, not main)
------------------------------------------------------------
# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Active

## Decisions Log
- [2026-08-30 18:26 UTC] Cosine similarity on the raw embedding, no re-ranking pass for v1.
- [2026-08-30 18:26 UTC] Added a basic integration test for the ranking function.

## Sub-Branches

## Open Questions

------------------------------------------------------------
search-endpoint's Sub-Branches line for it is now Completed:
# Branch Memory: search-endpoint

## Branched From
main

## Goal
Add a /search endpoint backed by the new embeddings index.

## Status
Active

## Decisions Log
- [2026-08-30 18:26 UTC] Chose FAISS over a hosted vector DB — no external d

In [8]:
# Now close the parent. It merges into main, since main is *its* base - this
# is the one that actually gets a STATE_TRACKER.md row, and Ollama sees the
# ranking branch's rolled-up Sub-Branches summary too, not just the parent's
# own notes.
try:
    archived_parent = tools.close_branch("search-endpoint")
    print(f"Archived to: {archived_parent}")
    print("-" * 60)
    print(subprocess.run(["git", "show", "main:STATE_TRACKER.md"], cwd=sandbox, capture_output=True, text=True, check=True).stdout)
except tools.GitAgentError as exc:
    print(f"close_branch failed (is Ollama running? `ollama serve`): {exc}")

Archived to: C:\Users\user\AppData\Local\Temp\gitagent_demo_yxvtla6g\branches\archived\search-endpoint\MEMORY.md
------------------------------------------------------------
# Demo State Tracker

## Side Branches (Features & Quests)

| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |
|---|---|---|---|---|
| Branch-001 | search-endpoint | Add a /search endpoint backed by the new embeddings index. | Completed | Here is a summary of the branch memory in 2-3 plain-text sentences:  This branch added a /search endpoint backed by the new embeddings index and made the key decision to use cosine similarity on raw embeddings without re-ranking for the initial version. The branch also completed a sub-branch focused on ranking functionality, including a basic integration test. The outcome is a search endpoint ranking formula ready for testing. |



## 4. `render_branch_graph` — see the branch structure at a glance

No LLM call — it just reads `git log --all` and lays commits out into lanes the way `git log --graph` does, then renders SVG. With real nested branches now merged in, this graph actually has more than one lane. Returns the path to a standalone HTML file; open it in a browser to see the colored graph + commit log side by side.

In [9]:
graph_path = visualize.render_branch_graph()
print(f"Graph written to: {graph_path}")
print("Open it in a browser to view — it's a plain, self-contained HTML file.")

Graph written to: C:\Users\user\AppData\Local\Temp\gitagent_demo_yxvtla6g\branch_graph.html
Open it in a browser to view — it's a plain, self-contained HTML file.


## The real git history behind it

Every step above was a real commit, and the merges are now real merges — the nested branch's commits genuinely diverged before being folded back in. Look for the `Log open of ... under ...` / `Log close of ... under ...` commits - those are the Sub-Branches bookkeeping, landing on the root branch's own history rather than on `main`.

In [10]:
log = subprocess.run(
    ["git", "log", "--oneline", "--all", "--graph"],
    cwd=sandbox,
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout)

* 1321565 Close branch: search-endpoint
* edd8fa9 Archive branch: search-endpoint
*   8beaeaf Merge branch 'search-endpoint'
|\  
| * 12fd339 Log close of search-endpoint-ranking under search-endpoint
| * 797d785 Archive branch: search-endpoint-ranking
| *   e08d0c2 Merge branch 'search-endpoint-ranking'
| |\  
| | * fd47caa Update branch: search-endpoint-ranking
| | * 9e0da61 Update branch: search-endpoint-ranking
| | * 5ccf16e Open branch: search-endpoint-ranking
| * | 17150fe Update branch: search-endpoint
| |/  
| * 5f17848 Log open of search-endpoint-ranking under search-endpoint
| * 5dca7b5 Open branch: search-endpoint
|/  
* 9d68963 Open branch: search-endpoint
* 6e5c69f Initial commit



## Using the CLI against the real project

The same four operations are exposed as CLI subcommands, run from the repo root:

```bash
python -m gitagent.cli open-branch <name> "<description>" [-b BASE]
python -m gitagent.cli update-branch <name> "<note>"
python -m gitagent.cli close-branch <name>
python -m gitagent.cli graph [-o OUTPUT]
```

`-b/--base` defaults to `main`; pass another open branch's name to nest under it, e.g. `python -m gitagent.cli open-branch search-endpoint-ranking "..." -b search-endpoint`.

## Roadmap / open questions (from `CLAUDE.md`)

- Whether `close_branch`'s summarization should stay on the local model or become a hybrid call to Claude if local quality proves insufficient.
- A future `search_branches` tool over archived branch memories via embeddings (the RAG learning branch).
- Final tech stack for orchestration/vector store (tracked as Branch-003 in `STATE_TRACKER.md`).

In [11]:
shutil.rmtree(sandbox, ignore_errors=True)
print(f"Cleaned up sandbox: {sandbox}")

Cleaned up sandbox: C:\Users\user\AppData\Local\Temp\gitagent_demo_yxvtla6g
